[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_02/NB_bishop_ch02_probability.ipynb)

# Chapter 2: Probabilities

This notebook covers foundational probability concepts from Bishop's *Deep Learning: Foundations and Concepts*, Chapter 2: Bayes' theorem, Gaussian distributions, entropy, KL divergence, and maximum likelihood estimation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)

## 1. Bayes' Theorem: Medical Screening Example

A disease has prevalence $P(D) = 0.01$. A test has:
- Sensitivity (true positive rate): $P(+|D) = 0.95$
- Specificity (true negative rate): $P(-|\neg D) = 0.95$

By Bayes' theorem:
$$P(D|+) = \frac{P(+|D)\,P(D)}{P(+|D)\,P(D) + P(+|\neg D)\,P(\neg D)}$$

In [ ]:
prevalence = 0.01
sensitivity = 0.95
specificity = 0.95

p_pos_given_d = sensitivity
p_pos_given_nd = 1 - specificity
p_d = prevalence
p_nd = 1 - prevalence

p_d_given_pos = (p_pos_given_d * p_d) / (p_pos_given_d * p_d + p_pos_given_nd * p_nd)

print(f'P(Disease | Positive test) = {p_d_given_pos:.4f}')
print(f'\nDespite 95% test accuracy, only {p_d_given_pos*100:.1f}% of positive results')
print(f'indicate actual disease when prevalence is {prevalence*100:.0f}%.')

In [ ]:
prevalences = np.linspace(0.001, 0.5, 200)
specificities = [0.90, 0.95, 0.99]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#cd0000', '#1a3a6e', '#228B22']

for spec, c in zip(specificities, colors):
    fp_rate = 1 - spec
    ppv = (sensitivity * prevalences) / (
        sensitivity * prevalences + fp_rate * (1 - prevalences))
    ax.plot(prevalences * 100, ppv * 100, linewidth=2, color=c,
            label=f'Specificity = {spec:.0%}')

ax.set_xlabel('Prevalence (%)')
ax.set_ylabel('Positive Predictive Value (%)')
ax.set_title("Bishop Ch 2: Bayes' Theorem -- Medical Screening")
ax.legend()
fig.tight_layout()
save_fig(fig, 'bishop_ch2_bayes_screening')
plt.show()

## 2. Prior, Likelihood, and Posterior

Visualize how the posterior is proportional to prior times likelihood.

In [ ]:
theta = np.linspace(0, 1, 500)

# Prior: Beta(2, 5)
a_prior, b_prior = 2, 5
prior = stats.beta.pdf(theta, a_prior, b_prior)

# Data: 6 successes in 10 trials
n, k = 10, 6
likelihood = theta ** k * (1 - theta) ** (n - k)
likelihood /= likelihood.max()  # normalize for plotting

# Posterior: Beta(a + k, b + n - k)
a_post, b_post = a_prior + k, b_prior + n - k
posterior = stats.beta.pdf(theta, a_post, b_post)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(theta, prior / prior.max(), '--', color='#1a3a6e', linewidth=2,
        label=f'Prior: Beta({a_prior},{b_prior})')
ax.plot(theta, likelihood, ':', color='#FF8C00', linewidth=2,
        label=f'Likelihood (k={k}, n={n})')
ax.plot(theta, posterior / posterior.max(), '-', color='#cd0000', linewidth=2,
        label=f'Posterior: Beta({a_post},{b_post})')
ax.set_xlabel(r'$\theta$')
ax.set_ylabel('Density (scaled)')
ax.set_title('Prior x Likelihood = Posterior')
ax.legend()
fig.tight_layout()
plt.show()

## 3. The Gaussian Distribution

In [ ]:
x = np.linspace(-5, 5, 500)

params = [(0, 0.5), (0, 1.0), (0, 2.0), (-2, 1.0)]
colors_gauss = ['#1a3a6e', '#cd0000', '#228B22', '#FF8C00']

fig, ax = plt.subplots(figsize=(8, 5))
for (mu, sigma), c in zip(params, colors_gauss):
    y = stats.norm.pdf(x, mu, sigma)
    ax.plot(x, y, linewidth=2, color=c,
            label=rf'$\mu={mu}, \sigma={sigma}$')

ax.set_xlabel('x')
ax.set_ylabel('p(x)')
ax.set_title('Bishop Ch 2: Gaussian Distribution')
ax.legend()
fig.tight_layout()
save_fig(fig, 'bishop_ch2_gaussian')
plt.show()

## 4. Maximum Likelihood Estimation for Gaussian

Given data $\{x_1, \ldots, x_N\}$, the MLE estimates are:
$$\hat{\mu}_{\text{ML}} = \frac{1}{N}\sum_n x_n, \qquad \hat{\sigma}^2_{\text{ML}} = \frac{1}{N}\sum_n (x_n - \hat{\mu})^2$$

In [ ]:
np.random.seed(42)
true_mu, true_sigma = 3.0, 1.5
data = np.random.normal(true_mu, true_sigma, 50)

mu_ml = np.mean(data)
sigma_ml = np.std(data)  # biased MLE
sigma_unbiased = np.std(data, ddof=1)

print(f'True parameters:     mu = {true_mu:.2f}, sigma = {true_sigma:.2f}')
print(f'MLE estimates:       mu = {mu_ml:.4f}, sigma = {sigma_ml:.4f}')
print(f'Unbiased sigma:      sigma = {sigma_unbiased:.4f}')

In [ ]:
# Show how MLE improves with more data
sample_sizes = np.arange(5, 501, 5)
mu_estimates = []
sigma_estimates = []

np.random.seed(42)
full_data = np.random.normal(true_mu, true_sigma, 500)

for n in sample_sizes:
    subset = full_data[:n]
    mu_estimates.append(np.mean(subset))
    sigma_estimates.append(np.std(subset))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(sample_sizes, mu_estimates, color='#1a3a6e', linewidth=1.5)
ax1.axhline(true_mu, color='#cd0000', linestyle='--', linewidth=1.5, label=r'True $\mu$')
ax1.set_xlabel('Sample size N')
ax1.set_ylabel(r'$\hat{\mu}_{ML}$')
ax1.set_title('MLE of Mean')
ax1.legend()

ax2.plot(sample_sizes, sigma_estimates, color='#1a3a6e', linewidth=1.5)
ax2.axhline(true_sigma, color='#cd0000', linestyle='--', linewidth=1.5, label=r'True $\sigma$')
ax2.set_xlabel('Sample size N')
ax2.set_ylabel(r'$\hat{\sigma}_{ML}$')
ax2.set_title('MLE of Std Dev')
ax2.legend()

fig.suptitle('MLE Convergence', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## 5. Entropy

Shannon entropy of a discrete distribution:
$$H[p] = -\sum_x p(x) \ln p(x)$$

In [ ]:
def entropy(p):
    """Shannon entropy in nats."""
    p = np.array(p)
    p = p[p > 0]
    return -np.sum(p * np.log(p))

# Binary entropy as function of p
p_vals = np.linspace(0.001, 0.999, 500)
h_binary = -p_vals * np.log(p_vals) - (1 - p_vals) * np.log(1 - p_vals)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(p_vals, h_binary, color='#1a3a6e', linewidth=2)
ax.set_xlabel('p')
ax.set_ylabel('H(p)')
ax.set_title('Binary Entropy Function')
ax.axvline(0.5, color='#cd0000', linestyle='--', alpha=0.6, label='Maximum at p=0.5')
ax.legend()
fig.tight_layout()
plt.show()

## 6. KL Divergence

$$\text{KL}(p \| q) = -\sum_x p(x) \ln \frac{q(x)}{p(x)} = \sum_x p(x) \ln \frac{p(x)}{q(x)}$$

We compare two Gaussian distributions and visualize KL divergence.

In [ ]:
def kl_gaussian(mu_p, sigma_p, mu_q, sigma_q):
    """Analytical KL divergence between two univariate Gaussians."""
    return (np.log(sigma_q / sigma_p) +
            (sigma_p ** 2 + (mu_p - mu_q) ** 2) / (2 * sigma_q ** 2) - 0.5)

# KL divergence as we vary mu_q
mu_p, sigma_p = 0, 1
mu_q_range = np.linspace(-3, 3, 200)
sigma_q_range = np.linspace(0.3, 3.0, 200)

kl_mu = [kl_gaussian(mu_p, sigma_p, mq, sigma_p) for mq in mu_q_range]
kl_sigma = [kl_gaussian(mu_p, sigma_p, mu_p, sq) for sq in sigma_q_range]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(mu_q_range, kl_mu, color='#1a3a6e', linewidth=2)
ax1.set_xlabel(r'$\mu_q$')
ax1.set_ylabel(r'$\mathrm{KL}(p \| q)$')
ax1.set_title(r'KL Divergence vs $\mu_q$ ($\sigma_p = \sigma_q = 1$)')

ax2.plot(sigma_q_range, kl_sigma, color='#cd0000', linewidth=2)
ax2.set_xlabel(r'$\sigma_q$')
ax2.set_ylabel(r'$\mathrm{KL}(p \| q)$')
ax2.set_title(r'KL Divergence vs $\sigma_q$ ($\mu_p = \mu_q = 0$)')

fig.suptitle('Bishop Ch 2: KL Divergence Between Gaussians', fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch2_kl_divergence')
plt.show()

## 7. KL Divergence is Asymmetric

In [ ]:
mu_p, sigma_p = 0, 1
mu_q, sigma_q = 2, 0.5

kl_pq = kl_gaussian(mu_p, sigma_p, mu_q, sigma_q)
kl_qp = kl_gaussian(mu_q, sigma_q, mu_p, sigma_p)

print(f'KL(p || q) = {kl_pq:.4f}')
print(f'KL(q || p) = {kl_qp:.4f}')
print(f'\nKL divergence is NOT symmetric: KL(p||q) != KL(q||p)')

x = np.linspace(-4, 5, 500)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, stats.norm.pdf(x, mu_p, sigma_p), color='#1a3a6e', linewidth=2,
        label=f'p: N({mu_p}, {sigma_p})')
ax.plot(x, stats.norm.pdf(x, mu_q, sigma_q), color='#cd0000', linewidth=2,
        label=f'q: N({mu_q}, {sigma_q})')
ax.fill_between(x, stats.norm.pdf(x, mu_p, sigma_p), alpha=0.15, color='#1a3a6e')
ax.fill_between(x, stats.norm.pdf(x, mu_q, sigma_q), alpha=0.15, color='#cd0000')
ax.set_xlabel('x')
ax.set_ylabel('Density')
ax.set_title(f'KL(p||q) = {kl_pq:.3f},   KL(q||p) = {kl_qp:.3f}')
ax.legend()
fig.tight_layout()
plt.show()

## 8. Mutual Information

In [ ]:
# Joint distribution example
joint = np.array([[0.1, 0.05, 0.02],
                  [0.05, 0.3, 0.08],
                  [0.02, 0.08, 0.3]])
joint /= joint.sum()

px = joint.sum(axis=1)
py = joint.sum(axis=0)
independent = np.outer(px, py)

# Mutual information
mi = 0
for i in range(3):
    for j in range(3):
        if joint[i, j] > 0:
            mi += joint[i, j] * np.log(joint[i, j] / independent[i, j])

print(f'Mutual Information I(X;Y) = {mi:.4f} nats')
print(f'H(X) = {entropy(px):.4f}, H(Y) = {entropy(py):.4f}')
print(f'H(X,Y) = {entropy(joint.ravel()):.4f}')
print(f'I(X;Y) = H(X) + H(Y) - H(X,Y) = {entropy(px) + entropy(py) - entropy(joint.ravel()):.4f}')

## 9. Summary

**Key takeaways from Bishop Chapter 2:**

- Bayes' theorem converts prior beliefs into posterior probabilities given evidence.
- The base rate (prevalence) dramatically affects diagnostic conclusions.
- Maximum likelihood provides point estimates that converge with more data.
- Entropy measures uncertainty; KL divergence measures distributional mismatch.
- KL divergence is asymmetric, which has implications for variational inference.